# 04 — Évaluation finale sur le jeu de test

Test = steps 632 à 743, la période la plus récente. Il n'a servi à aucun choix :
modèle, variables, poids, hyperparamètres et seuil ont tous été fixés sur la validation.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("fraude-04-test")
         .master("spark://spark-master:7077")
         .config("spark.driver.host", "spark-master")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .config("spark.sql.shuffle.partitions", "18")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.eventLog.enabled", "true")
         .config("spark.eventLog.dir", "file:///tmp/spark-events")
         .getOrCreate())

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/28 12:02:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

test = spark.read.parquet("/workspace/data/ml/test.parquet")
modele = PipelineModel.load("/workspace/data/modeles/modele_fraude")

with open("/workspace/data/modeles/seuil.txt") as f:
    seuil = float(f.read())

nb = test.count()
nb_fraudes = test.filter("label = 1").count()
print("Test :", nb, "lignes,", nb_fraudes, "fraudes")
print("PR-AUC du hasard (taux de fraude) :", round(nb_fraudes / nb, 4))
print("Seuil utilisé :", seuil)

Test : 88440 lignes, 1235 fraudes
PR-AUC du hasard (taux de fraude) : 0.014
Seuil utilisé : 0.7


La période de test contient plus de fraudes que l'entraînement. Le hasard donnerait une PR-AUC de 0,014.

In [3]:
# probabilité de fraude, puis prédiction avec le seuil choisi en validation
predictions = modele.transform(test)
predictions = predictions.withColumn("p_fraude", vector_to_array("probability")[1])
predictions = predictions.withColumn("prediction_seuil", (F.col("p_fraude") >= seuil).cast("double"))
predictions.cache()

pr_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(predictions)
print("PR-AUC :", round(pr_auc, 4))

for colonne in ["prediction", "prediction_seuil"]:
    print("\nPrédiction :", colonne)
    for metrique in ["precisionByLabel", "recallByLabel", "fMeasureByLabel"]:
        evaluateur = MulticlassClassificationEvaluator(labelCol="label", predictionCol=colonne,
                                                       metricName=metrique, metricLabel=1.0)
        print("  ", metrique, ":", round(evaluateur.evaluate(predictions), 4))

PR-AUC : 0.6779

Prédiction : prediction


   precisionByLabel : 0.8615
   recallByLabel : 0.4534
   fMeasureByLabel : 0.5942

Prédiction : prediction_seuil
   precisionByLabel : 0.9149
   recallByLabel : 0.3919
   fMeasureByLabel : 0.5488


- Le modèle fait beaucoup mieux que le hasard sur des données futures qu'il n'a jamais vues.
- Avec le seuil 0,7, ses alertes sont presque toujours justes, mais il ne détecte qu'une partie des fraudes.
- Sur cette période, un seuil plus bas aurait été un peu meilleur : le meilleur seuil change quand la proportion de fraudes change. On garde quand même 0,7, car le test ne doit servir à aucun choix.

## Matrice de confusion (lignes = vraie classe, colonnes = classe prédite)

In [4]:
predictions.groupBy("label").pivot("prediction_seuil", [0.0, 1.0]).count().na.fill(0).orderBy("label").show()

+-----+-----+---+
|label|  0.0|1.0|
+-----+-----+---+
|  0.0|87160| 45|
|  1.0|  751|484|
+-----+-----+---+



Très peu de fausses alertes, mais une bonne partie des fraudes passe encore : le modèle est prudent.

## Résultats par type de transaction

In [5]:
(predictions.groupBy("type")
            .agg(F.count("*").alias("nb"),
                 F.sum("label").alias("fraudes"),
                 F.sum(F.col("prediction_seuil") * F.col("label")).alias("fraudes_detectees"),
                 F.sum(F.col("prediction_seuil") * (1 - F.col("label"))).alias("fausses_alertes"))
            .orderBy(F.desc("fraudes"))
            .show())

[Stage 66:================================>                         (5 + 4) / 9]

+--------+-----+-------+-----------------+---------------+
|    type|   nb|fraudes|fraudes_detectees|fausses_alertes|
+--------+-----+-------+-----------------+---------------+
|CASH_OUT|28824|  620.0|            255.0|            8.0|
|TRANSFER| 8711|  615.0|            229.0|           37.0|
|   DEBIT|  707|    0.0|              0.0|            0.0|
| CASH_IN|20484|    0.0|              0.0|            0.0|
| PAYMENT|29714|    0.0|              0.0|            0.0|
+--------+-----+-------+-----------------+---------------+



Les fraudes sont détectées de la même façon dans `CASH_OUT` et `TRANSFER`, et il n'y a aucune fausse alerte sur les autres types : le modèle a bien compris le rôle du type.

## Importance des features

In [6]:
dernier_etage = modele.stages[-1]

# noms des features, lus dans les métadonnées du VectorAssembler
noms = {}
for groupe in predictions.schema["features_raw"].metadata["ml_attr"]["attrs"].values():
    for attribut in groupe:
        noms[attribut["idx"]] = attribut["name"]

if hasattr(dernier_etage, "featureImportances"):        # Random Forest / GBT
    valeurs = dernier_etage.featureImportances.toArray()
else:                                                   # régression logistique
    valeurs = dernier_etage.coefficients.toArray()

for i in sorted(noms, key=lambda i: -abs(valeurs[i])):
    print(f"{noms[i]:<20} {valeurs[i]:.4f}")

oldbalanceDest       0.3502
heure                0.2389
logAmount            0.1972
typeVec_CASH_OUT     0.1415
typeVec_TRANSFER     0.0565
typeVec_CASH_IN      0.0087
typeVec_PAYMENT      0.0070


Le solde du destinataire, l'heure et le montant sont les variables les plus utiles. Aucune ne domine seule : le modèle ne dépend plus d'un raccourci du simulateur.

## Conclusion

- Les données brutes ont été nettoyées avec Spark (SQL, DataFrame, `Imputer`).
- Pipeline final : `Imputer` → `SQLTransformer` → `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler` → GBT, avec un poids pour les fraudes et un seuil choisi sur la validation.
- Le modèle n'utilise que des informations connues au moment de la transaction.
- Sur le test, il fait beaucoup mieux que le hasard, avec très peu de fausses alertes.
- Limite : la proportion de fraudes change selon la période, ce qui rend le choix du seuil moins stable.

In [7]:
spark.stop()